In [1]:
import pandas as pd
import numpy as np
import keras
from keras import layers, ops
from sklearn.model_selection import KFold

import ast

In [2]:
def prepare_melodies(melodies, max_len=128, pitch_to_idx=None):
    """
    Convert list of pitch sequences into transformer-ready format.

    For each melody [p0, p1, p2, ..., pN]:
      - Input:  [PAD, PAD, ..., p0, p1, ..., p_{N-1}]
      - Target: [PAD, PAD, ..., p1, p2, ..., pN]

    Both padded/truncated to max_len.

    Args:
        melodies: list of lists, each containing MIDI pitch integers
        max_len: maximum sequence length (window size)
        pitch_to_idx: optional pre-existing vocabulary mapping.
                      If None, vocabulary is built from the melodies.
                      If provided (e.g., from training set), it is reused.
                      This ensures test data uses the same vocabulary as training.

    Returns:
        xs: input array (batch, max_len)
        ys: target array (batch, max_len)
        mask: boolean array (batch, max_len) - True where loss should be computed
        vocab_size: number of unique pitches + 1 (for PAD)
        pitch_to_idx: mapping from MIDI pitch to model index
        idx_to_pitch: mapping from model index to MIDI pitch
    """
    # Build or reuse vocabulary
    if pitch_to_idx is None:
        all_pitches = sorted(set(p for mel in melodies for p in mel))
        pitch_to_idx = {p: i + 1 for i, p in enumerate(all_pitches)}

    idx_to_pitch = {i: p for p, i in pitch_to_idx.items()}
    vocab_size = max(pitch_to_idx.values()) + 1  # +1 because index 0 is PAD

    PAD = 0

    xs = []
    ys = []
    masks = []
    skipped = 0

    for mel in melodies:
        # Convert to indices, skipping pitches not in vocabulary
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]

        if len(indexed) < 2:
            skipped += 1
            continue

        # Input: all tokens except the last
        # Target: all tokens except the first (shifted by 1)
        inp = indexed[:-1]
        tgt = indexed[1:]

        # Pad or truncate to max_len
        if len(inp) >= max_len:
            # Truncate: take last max_len tokens
            inp = inp[-max_len:]
            tgt = tgt[-max_len:]
            m = [True] * max_len
        else:
            # Pad from the left
            pad_len = max_len - len(inp)
            m = [False] * pad_len + [True] * len(inp)
            inp = [PAD] * pad_len + inp
            tgt = [PAD] * pad_len + tgt

        xs.append(inp)
        ys.append(tgt)
        masks.append(m)

    if skipped > 0:
        print(f"  Warning: skipped {skipped} melodies (too short or unknown pitches)")

    return (np.array(xs), np.array(ys), np.array(masks),
            vocab_size, pitch_to_idx, idx_to_pitch)

In [5]:
# Model components
class TokenAndPositionEmbedding(layers.Layer):
    """Embeds tokens and adds learned positional encoding."""

    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=False
        )
        self.pos_emb = layers.Embedding(
            input_dim=max_len, output_dim=embed_dim
        )

    def call(self, x):
        seq_len = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=seq_len, step=1)
        token_embeddings = self.token_emb(x)
        position_embeddings = self.pos_emb(positions)
        return token_embeddings + position_embeddings


class CausalTransformerBlock(layers.Layer):
    """
    Transformer block with CAUSAL attention mask.

    The causal mask ensures position i can only attend to positions <= i,
    matching IDyOM's left-to-right prediction constraint.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads,
#             dropout=dropout_rate
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Causal mask: each position can only attend to itself and earlier positions
        attn_output = self.att(
            inputs, inputs, use_causal_mask=True
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [6]:
def build_transformer(
    vocab_size,
    max_len=64,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
):
    """
    Build a simple causal transformer for next-pitch prediction.

    Default configuration is deliberately small for fair comparison with IDyOM:
    - embed_dim=32, num_heads=4, ff_dim=64, num_layers=2
    """
    inputs = keras.Input(shape=(max_len,))

    # Embedding
    x = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)(inputs)

    # Transformer blocks with causal masking
    for _ in range(num_layers):
        x = CausalTransformerBlock(embed_dim, num_heads, ff_dim, dropout_rate)(x)

    # Per-position prediction head
    # Dense applied to (batch, seq_len, embed_dim) works per-position automatically
    outputs = layers.Dense(vocab_size, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [7]:
def masked_sparse_crossentropy(y_true, y_pred):
    """
    Sparse categorical crossentropy that ignores PAD tokens (index 0).

    This ensures padding doesn't artificially lower the loss.
    """
    # Create mask: True where target is not PAD
    mask = ops.cast(y_true != 0, dtype="float32")

    # Compute per-position loss
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)

    # Apply mask and average over non-padded positions only
    masked_loss = loss * mask
    return ops.sum(masked_loss) / (ops.sum(mask) + 1e-8)


In [8]:
def compute_ic(model, xs, ys, masks):
    """
    Compute Information Content (IC) in bits for each note prediction.

    Args:
        model: trained transformer model
        xs: input sequences (batch, max_len)
        ys: target sequences (batch, max_len)
        masks: boolean mask (batch, max_len) - True for real notes

    Returns:
        mean_ic: mean IC in bits across all predicted notes
        all_ics: list of IC values per note (flattened)
    """
    # Get model predictions
    probs = model.predict(xs, verbose=0)

    all_ics = []

    for i in range(len(xs)):
        for t in range(len(xs[i])):
            if masks[i][t]:
                # Probability assigned to the correct next token
                true_token = ys[i][t]
                p = probs[i][t][true_token]

                # IC in bits (p is a clip to avoid log(0))
                p = max(p, 1e-10)
                ic = -np.log2(p)
                all_ics.append(ic)

    mean_ic = np.mean(all_ics)
    return mean_ic, all_ics


def compute_ic_per_melody(model, xs, ys, masks, melody_indices):
    """
    Compute mean IC per melody (useful for per-piece comparison with IDyOM).

    Args:
        melody_indices: list of melody names/IDs corresponding to each row in xs

    Returns:
        dict mapping melody_id -> mean IC in bits
    """
    probs = model.predict(xs, verbose=0)
    melody_ics = {}

    for i in range(len(xs)):
        mel_id = melody_indices[i]
        ics = []
        for t in range(len(xs[i])):
            if masks[i][t]:
                true_token = ys[i][t]
                p = max(probs[i][t][true_token], 1e-10)
                ics.append(-np.log2(p))
        melody_ics[mel_id] = np.mean(ics)

    return melody_ics


In [9]:
def train_model(
    melodies,
    max_len=128,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    validation_split=0.1,
    patience=5,
):
    """
    Simple full-corpus training (for initial testing only).
    For proper evaluation, use run_kfold or run_cross_corpus.
    """
    xs, ys, masks, vocab_size, pitch_to_idx, idx_to_pitch = \
        prepare_melodies(melodies, max_len=max_len)

    print(f"Data prepared:")
    print(f"  Melodies: {len(melodies)}")
    print(f"  Vocab size: {vocab_size} ({vocab_size - 1} pitches + PAD)")
    print(f"  Sequence length: {max_len}")
    print(f"  Input shape: {xs.shape}")

    model = build_transformer(
        vocab_size=vocab_size, max_len=max_len, embed_dim=embed_dim,
        num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
        dropout_rate=dropout_rate,
    )

    print(f"\n  Total parameters: {model.count_params():,}")
    model.summary()

    model.compile(
        loss=masked_sparse_crossentropy,
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True,
    )

    history = model.fit(
        xs, ys,
        sample_weight=masks.astype(np.float32),
        batch_size=batch_size, epochs=epochs,
        validation_split=validation_split, callbacks=[early_stop],
    )

    return model, history, {
        "vocab_size": vocab_size,
        "pitch_to_idx": pitch_to_idx,
        "idx_to_pitch": idx_to_pitch,
        "max_len": max_len,
    }

In [10]:
#     IDyOM command per fold:
# (idyom:idyom <test-dataset-id> '(cpitch) '(cpitch) :models :ltm :pretraining-ids '(<train-dataset-id>))

def run_kfold(
    melodies,
    melody_ids=None,
    k=10,
    max_len=128,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    patience=5,
    random_state=42,
    export_folds_path=None,
):
    """
    K-fold cross-validation for fair comparison with IDyOM.

    Both models must use identical train/test splits. This function:
    1. Splits melodies into k folds at the piece level
    2. For each fold, trains on training melodies, evaluates on test melodies
    3. Exports fold assignments so IDyOM can use the same splits

    The exported CSV has columns: melody_id, fold, split
    Use it to create matching IDyOM datasets for each fold.
    """
    if melody_ids is None:
        melody_ids = list(range(len(melodies)))

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)

    all_ics = []
    fold_results = []
    fold_assignments = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(melodies)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{k}")
        print(f"{'='*60}")

        train_melodies = [melodies[i] for i in train_idx]
        test_melodies = [melodies[i] for i in test_idx]
        train_ids = [melody_ids[i] for i in train_idx]
        test_ids = [melody_ids[i] for i in test_idx]

        print(f"Train: {len(train_melodies)} melodies")
        print(f"Test:  {len(test_melodies)} melodies")

        # Record fold assignments
        for idx in train_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "train",
            })
        for idx in test_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "test",
            })

        # Prepare data: vocabulary built from training set only
        xs_train, ys_train, masks_train, vocab_size, pitch_to_idx, _ = \
            prepare_melodies(train_melodies, max_len=max_len)

        # Test data uses same vocabulary
        xs_test, ys_test, masks_test, _, _, _ = \
            prepare_melodies(test_melodies, max_len=max_len, pitch_to_idx=pitch_to_idx)

        print(f"Vocab size: {vocab_size}")

        # Build and train
        model = build_transformer(
            vocab_size=vocab_size, max_len=max_len, embed_dim=embed_dim,
            num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
            dropout_rate=dropout_rate,
        )

        if fold == 0:
            print(f"Total parameters: {model.count_params():,}")

        model.compile(
            loss=masked_sparse_crossentropy,
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        )

        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )

        model.fit(
            xs_train, ys_train,
            sample_weight=masks_train.astype(np.float32),
            batch_size=batch_size, epochs=epochs,
            validation_split=0.1,  # within training set, for early stopping only
            callbacks=[early_stop], verbose=1,
        )

        # Evaluate on test fold
        mean_ic, ics = compute_ic(model, xs_test, ys_test, masks_test)
        melody_ics = compute_ic_per_melody(model, xs_test, ys_test, masks_test, test_ids)

        print(f"\nFold {fold + 1} mean IC: {mean_ic:.3f} bits")
        all_ics.extend(ics)
        fold_results.append({
            "fold": fold + 1,
            "mean_ic": mean_ic,
            "train_size": len(train_melodies),
            "test_size": len(test_melodies),
            "melody_ics": melody_ics,
        })

    # Export fold assignments for IDyOM replication
    df_folds = pd.DataFrame(fold_assignments)
    if export_folds_path:
        df_folds.to_csv(export_folds_path, index=False)
        print(f"\nFold assignments saved to: {export_folds_path}")

    # Summary
    overall_mean_ic = np.mean(all_ics)
    overall_std_ic = np.std(all_ics)

    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS ({k}-fold cross-validation)")
    print(f"{'='*60}")
    print(f"Mean IC: {overall_mean_ic:.3f} bits")
    print(f"Std IC:  {overall_std_ic:.3f} bits")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['mean_ic']:.3f} bits ({r['test_size']} test melodies)")

    return {
        "overall_mean_ic": overall_mean_ic,
        "overall_std_ic": overall_std_ic,
        "all_ics": all_ics,
        "fold_results": fold_results,
        "fold_assignments": df_folds,
    }

## Essen

In [ ]:
essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
max_len = max(len(m) for m in essen_melodies)

# model, history, data_info = train_model(
#     melodies,
#     max_len=128,
#     embed_dim=32,
#     num_heads=4,
#     ff_dim=64,
#     num_layers=2,
#     epochs=10,
# )

model, history, data_info = train_model(essen_melodies, max_len=max_len, epochs=50)


In [ ]:
# Compute IC
xs, ys, masks, _, _, _ = prepare_melodies(essen_melodies, max_len=128)
mean_ic, all_ics = compute_ic(model, xs, ys, masks)
print(f"\nMean IC: {mean_ic:.3f} bits")


Mean IC: 2.432 bits


In [ ]:
essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
essen_melody_ids = pd.read_csv("essen_unique_melodies.csv")["melody_id"].tolist()
max_len = max(len(m) for m in essen_melodies)

results = run_kfold(
    essen_melodies, melody_ids=essen_melody_ids, k=5,
    max_len=max_len, epochs=50,
    export_folds_path="full_essen_fold_assignments.csv",
)


FOLD 1/5
Train: 6562 melodies
Test:  1641 melodies
Vocab size: 52
Total parameters: 36,532
Epoch 1/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 27s 75ms/step - loss: 0.2407 - val_loss: 0.3110
Epoch 2/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1947 - val_loss: 0.2797
Epoch 3/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1860 - val_loss: 0.2695
Epoch 4/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1822 - val_loss: 0.2651
Epoch 5/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1799 - val_loss: 0.2596
Epoch 6/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1783 - val_loss: 0.2583
Epoch 7/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1770 - val_loss: 0.2557
Epoch 8/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1760 - val_loss: 0.2543
Epoch 9/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1752 - val_loss: 0.2524
Epoch 10/50
185/185 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.1744 - val_loss: 0.2525
Epoch 11/50
185/185 ━━━━━━━━━━━━━━━━

## Meertens

In [12]:
meertens_melodies = pd.read_csv("meertens_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()

meertens_melody_ids = pd.read_csv("meertens_unique_melodies.csv")["melody_id"].tolist()

max_len = max(len(m) for m in meertens_melodies)

In [ ]:
# model, history, data_info = train_model(
#     melodies,
#     max_len=128,
#     embed_dim=32,
#     num_heads=4,
#     ff_dim=64,
#     num_layers=2,
#     epochs=10,
# )

meertens_model, meertens_history, meertens_data_info = train_model(
    meertens_melodies,
    max_len=max_len,
    epochs=150
)

Data prepared:
  Melodies: 17620
  Vocab size: 46 (45 pitches + PAD)
  Sequence length: 180
  Input shape: (17620, 180)

  Total parameters: 50,990


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 180)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 180, 32)        │         7,232 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ causal_transformer_block        │ (None, 180, 32)        │        21,120 │
│ (CausalTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ causal_transformer_block_1      │ (None, 180, 32)        │        21,120 │
│ (CausalTransformerBlock)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 180, 46)        │         1,518 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,990 (199.18 KB)

 Trainable params: 50,990 (199.18 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 52s 53ms/step - loss: 0.7739 - val_loss: 0.6940
Epoch 2/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6999 - val_loss: 0.6791
Epoch 3/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6881 - val_loss: 0.6704
Epoch 4/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6819 - val_loss: 0.6659
Epoch 5/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6777 - val_loss: 0.6623
Epoch 6/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6741 - val_loss: 0.6588
Epoch 7/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6715 - val_loss: 0.6555
Epoch 8/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6685 - val_loss: 0.6533
Epoch 9/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6657 - val_loss: 0.6500
Epoch 10/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6634 - val_loss: 0.6479
Epoch 11/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.6610 - val_loss: 0.6442
Epoch 12/150
496/496 ━━━━━━━━━━━━━━━━━━

In [ ]:
# Compute IC
xs, ys, masks, _, _, _ = prepare_melodies(meertens_melodies, max_len=180)
mean_ic, all_ics = compute_ic(meertens_model, xs, ys, masks)
print(f"\nMean IC: {mean_ic:.3f} bits")


Mean IC: 2.178 bits


In [13]:
results = run_kfold(
    meertens_melodies, melody_ids=meertens_melody_ids, k=10,
    max_len=max_len, epochs=150,
    export_folds_path="full_meertens_fold_assignments.csv",
)


FOLD 1/10
Train: 15858 melodies
Test:  1762 melodies
Vocab size: 46
Total parameters: 59,662
Epoch 1/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 76s 143ms/step - loss: 0.1222 - val_loss: 0.1066
Epoch 2/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 64s 145ms/step - loss: 0.1080 - val_loss: 0.1034
Epoch 3/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 148ms/step - loss: 0.1058 - val_loss: 0.1022
Epoch 4/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1044 - val_loss: 0.1010
Epoch 5/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1033 - val_loss: 0.1002
Epoch 6/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1025 - val_loss: 0.0997
Epoch 7/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1020 - val_loss: 0.0992
Epoch 8/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1015 - val_loss: 0.0988
Epoch 9/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1012 - val_loss: 0.0987
Epoch 10/150
446/446 ━━━━━━━━━━━━━━━━━━━━ 66s 149ms/step - loss: 0.1009 - val_loss: 0.0983
Epoch

In [ ]:
folds = pd.read_csv("meertens_fold_assignments.csv")
folds

,melody_id,fold,split
0,0,1,train
1,1,1,train
2,2,1,train
3,4,1,train
4,5,1,train
...,...,...,...
176195,18056,10,test
176196,18070,10,test
176197,18086,10,test
176198,18092,10,test


In [ ]:
meta = pd.read_csv("meertens_unique_meta_melodies.csv")
meta

,melody_id,filename,path,pitch
0,0,NLB177881_01.krn,../data/meertens/NLB177881_01.krn,"[70, 72, 74, 70, 72, 69, 70, 62, 63, 62, 63, 6..."
1,1,NLB073485_01.krn,../data/meertens/NLB073485_01.krn,"[74, 74, 76, 74, 72, 71, 74, 67, 71, 72, 71, 6..."
2,2,NLB191040_01.krn,../data/meertens/NLB191040_01.krn,"[79, 79, 76, 77, 76, 74, 76, 76, 72, 69, 74, 7..."
3,3,NLB135827_01.krn,../data/meertens/NLB135827_01.krn,"[79, 81, 82, 81, 79, 77, 79, 74, 75, 72, 70, 7..."
4,4,NLB196963_01.krn,../data/meertens/NLB196963_01.krn,"[60, 64, 67, 72, 72, 71, 67, 64, 55, 55, 55, 7..."
...,...,...,...,...
17615,18103,NLB170249_01.krn,../data/meertens/NLB170249_01.krn,"[60, 60, 65, 65, 65, 65, 65, 65, 65, 65, 67, 6..."
17616,18104,NLB076423_01.krn,../data/meertens/NLB076423_01.krn,"[74, 74, 71, 67, 69, 62, 67, 74, 71, 74, 71, 7..."
17617,18105,NLB181046_01.krn,../data/meertens/NLB181046_01.krn,"[67, 67, 67, 72, 71, 74, 72, 71, 69, 72, 67, 6..."
17618,18106,NLB152233_01.krn,../data/meertens/NLB152233_01.krn,"[67, 74, 72, 74, 69, 70, 69, 67, 74, 79, 81, 8..."


In [ ]:
folds = folds.merge(meta, on="melody_id")
folds

,melody_id,fold,split,filename,path,pitch
0,0,1,train,NLB177881_01.krn,../data/meertens/NLB177881_01.krn,"[70, 72, 74, 70, 72, 69, 70, 62, 63, 62, 63, 6..."
1,1,1,train,NLB073485_01.krn,../data/meertens/NLB073485_01.krn,"[74, 74, 76, 74, 72, 71, 74, 67, 71, 72, 71, 6..."
2,2,1,train,NLB191040_01.krn,../data/meertens/NLB191040_01.krn,"[79, 79, 76, 77, 76, 74, 76, 76, 72, 69, 74, 7..."
3,4,1,train,NLB196963_01.krn,../data/meertens/NLB196963_01.krn,"[60, 64, 67, 72, 72, 71, 67, 64, 55, 55, 55, 7..."
4,5,1,train,NLB076099_01.krn,../data/meertens/NLB076099_01.krn,"[66, 67, 64, 67, 66, 67, 62, 67, 67, 66, 66, 6..."
...,...,...,...,...,...,...
176195,18056,10,test,NLB185929_01.krn,../data/meertens/NLB185929_01.krn,"[69, 74, 76, 76, 76, 77, 79, 77, 76, 77, 74, 7..."
176196,18070,10,test,NLB192050_01.krn,../data/meertens/NLB192050_01.krn,"[72, 74, 76, 77, 77, 79, 81, 81, 82, 79, 79, 7..."
176197,18086,10,test,NLB181146_01.krn,../data/meertens/NLB181146_01.krn,"[62, 66, 69, 74, 69, 71, 74, 73, 71, 69, 74, 6..."
176198,18092,10,test,NLB072359_01.krn,../data/meertens/NLB072359_01.krn,"[72, 74, 72, 71, 67, 69, 71, 69, 71, 69, 67, 6..."


In [ ]:
folds = folds.drop("pitch", axis=1)
folds

,melody_id,fold,split,filename,path
0,0,1,train,NLB177881_01.krn,../data/meertens/NLB177881_01.krn
1,1,1,train,NLB073485_01.krn,../data/meertens/NLB073485_01.krn
2,2,1,train,NLB191040_01.krn,../data/meertens/NLB191040_01.krn
3,4,1,train,NLB196963_01.krn,../data/meertens/NLB196963_01.krn
4,5,1,train,NLB076099_01.krn,../data/meertens/NLB076099_01.krn
...,...,...,...,...,...
176195,18056,10,test,NLB185929_01.krn,../data/meertens/NLB185929_01.krn
176196,18070,10,test,NLB192050_01.krn,../data/meertens/NLB192050_01.krn
176197,18086,10,test,NLB181146_01.krn,../data/meertens/NLB181146_01.krn
176198,18092,10,test,NLB072359_01.krn,../data/meertens/NLB072359_01.krn


In [1]:
folds.to_csv("meertens_fold_assignments.csv")

NameError: name 'folds' is not defined

## Train on Meertens, test on Essen

In [11]:
def run_cross_corpus(
    train_melodies,
    test_melodies,
    train_ids=None,
    test_ids=None,
    max_len=128,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=75,
    patience=5,
    validation_split=0.1,
):
    """
    Cross-corpus evaluation: train on one corpus, test on another.

    Vocabulary is built from the training corpus only. Test melodies
    with pitches outside the training vocabulary are skipped (same
    behavior as run_kfold for unseen pitches).
    """
    if train_ids is None:
        train_ids = list(range(len(train_melodies)))
    if test_ids is None:
        test_ids = list(range(len(test_melodies)))

    print(f"{'='*60}")
    print(f"CROSS-CORPUS: Train={len(train_melodies)}  Test={len(test_melodies)}")
    print(f"{'='*60}")

    # Vocabulary from training corpus only
    xs_train, ys_train, masks_train, vocab_size, pitch_to_idx, idx_to_pitch = \
        prepare_melodies(train_melodies, max_len=max_len)

    # Test corpus reuses training vocabulary
    xs_test, ys_test, masks_test, _, _, _ = \
        prepare_melodies(test_melodies, max_len=max_len, pitch_to_idx=pitch_to_idx)

    print(f"Vocab size (from train): {vocab_size}")
    print(f"Train shape: {xs_train.shape}  Test shape: {xs_test.shape}")

    model = build_transformer(
        vocab_size=vocab_size, max_len=max_len, embed_dim=embed_dim,
        num_heads=num_heads, ff_dim=ff_dim, num_layers=num_layers,
        dropout_rate=dropout_rate,
    )
    print(f"Total parameters: {model.count_params():,}")

    model.compile(
        loss=masked_sparse_crossentropy,
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True
    )

    history = model.fit(
        xs_train, ys_train,
        sample_weight=masks_train.astype(np.float32),
        batch_size=batch_size, epochs=epochs,
        validation_split=validation_split,
        callbacks=[early_stop], verbose=1,
    )

    mean_ic, all_ics = compute_ic(model, xs_test, ys_test, masks_test)
    melody_ics = compute_ic_per_melody(model, xs_test, ys_test, masks_test, test_ids)

    print(f"\nCross-corpus mean IC: {mean_ic:.3f} bits")
    print(f"Cross-corpus std  IC: {np.std(all_ics):.3f} bits")

    results = {
        "mean_ic": mean_ic,
        "std_ic": np.std(all_ics),
        "all_ics": all_ics,
        "melody_ics": melody_ics,
        "vocab_size": vocab_size,
        "pitch_to_idx": pitch_to_idx,
        "idx_to_pitch": idx_to_pitch,
        "history": history,
        "model": model,
    }

    return results

In [12]:
meertens = pd.read_csv("meertens_unique_melodies.csv")
essen = pd.read_csv("essen_unique_melodies.csv")

meertens_melodies = meertens["pitch"].apply(ast.literal_eval).tolist()
meertens_ids = meertens["melody_id"].tolist()

essen_melodies = essen["pitch"].apply(ast.literal_eval).tolist()
essen_ids = essen["melody_id"].tolist()

max_len = max(
    max(len(m) for m in meertens_melodies),
    max(len(m) for m in essen_melodies),
)

cc_results = run_cross_corpus(
    train_melodies=meertens_melodies,
    test_melodies=essen_melodies,
    train_ids=meertens_ids,
    test_ids=essen_ids,
    max_len=max_len,
    epochs=150,
)

CROSS-CORPUS: Train=17620  Test=8203
Vocab size (from train): 46
Train shape: (17620, 1237)  Test shape: (8203, 1237)
Total parameters: 59,662
Epoch 1/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 91s 157ms/step - loss: 0.1205 - val_loss: 0.1050
Epoch 2/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 71s 144ms/step - loss: 0.1072 - val_loss: 0.1021
Epoch 3/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - loss: 0.1051 - val_loss: 0.1009
Epoch 4/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - loss: 0.1039 - val_loss: 0.1000
Epoch 5/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - loss: 0.1030 - val_loss: 0.0994
Epoch 6/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - loss: 0.1024 - val_loss: 0.0991
Epoch 7/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 141ms/step - loss: 0.1018 - val_loss: 0.0985
Epoch 8/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 142ms/step - loss: 0.1015 - val_loss: 0.0982
Epoch 9/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 142ms/step - loss: 0.1012 - val_loss: 0.0978
Epoch 10/150
496/496 ━━━━━━━━━━━━━━━━━━━━ 70s 1